In [1]:
import time
import queue
import threading

In [2]:
class DynamicBatchServer:
    """
    지연(timeout) 기능을 포함한 동적 배치 서버를 시뮬레이션합니다.
    """
    def __init__(self, batch_size, timeout_ms):
        self.batch_size = batch_size
        self.timeout = timeout_ms / 1000.0  # ms를 초 단위로 변환
        self.request_queue = queue.Queue()
        self.server_running = True

    def add_request(self, request):
        """외부에서 들어온 사용자 요청을 대기 큐에 추가합니다."""
        print(f"[Client] >> 요청 {request['id']}({request['target_len']} 토큰) 서버에 도착.")
        self.request_queue.put(request)

    def _gather_requests(self, max_items):
        """
        타임아웃을 기다리며 대기 큐에서 요청을 수집합니다.
        """
        batch = []
        start_time = time.time()

        # 타임아웃이 되거나, 필요한 만큼 채우거나, 큐가 빌 때까지 요청을 모음
        while time.time() - start_time < self.timeout and \
              len(batch) < max_items and \
              not self.request_queue.empty():
            batch.append(self.request_queue.get())

        return batch

    def run(self):
        """서버의 메인 스케줄링 루프를 실행합니다."""
        active_batch = []
        step = 0

        while self.server_running:
            # 1. 빈자리가 있으면 새로운 요청을 배치에 추가 (Hop-on)
            needed = self.batch_size - len(active_batch)
            if needed > 0:
                 # 타임아웃을 기다리며 새로운 요청들을 가져옴
                 new_requests = self._gather_requests(max_items=needed)
                 if new_requests:
                    print(f"\n[Scheduler] >> 대기 큐에서 {[req['id'] for req in new_requests]}을(를) 배치에 추가!")
                    active_batch.extend(new_requests)

            # 처리할 배치가 없으면 루프 계속
            if not active_batch:
                time.sleep(0.01)
                # 실제 서버는 계속 실행되지만, 시뮬레이션에서는 종료 조건 확인
                if self.request_queue.empty():
                    # 외부에서 더 이상 요청을 추가하지 않으면 종료
                    pass
                continue

            step += 1
            # 2. 패딩: 현재 배치의 최대 길이를 다시 계산 (개념)
            max_len = max(req['target_len'] for req in active_batch)

            print(f"--- 스텝 {step} (현재 배치: {[req['id'] for req in active_batch]}, 최대 길이: {max_len}) ---")

            # 3. 토큰 생성 및 완료된 요청 확인
            completed_this_step = []
            for req in active_batch:
                if req['current_len'] < req['target_len']:
                    req['current_len'] += 1

                if req['current_len'] >= req['target_len']:
                    completed_this_step.append(req)

            time.sleep(0.2) # 각 스텝의 연산 시간 시뮬레이션

            # 4. 완료된 요청을 배치에서 제거하고 결과 반환 (Hop-off)
            if completed_this_step:
                print(f"\n[Scheduler] << 요청 {[req['id'] for req in completed_this_step]} 완료! 결과 반환.")
                active_batch = [req for req in active_batch if req not in completed_this_step]

            # 모든 요청이 처리되었고 큐도 비어있으면 시뮬레이션 종료
            if not active_batch and self.request_queue.empty():
                self.server_running = False

        print("\n--- 모든 요청 처리 완료 ---")

In [3]:
# --- 시뮬레이션 실행 ---
# 최대 배치크기 4, 300ms 타임아웃 설정
server = DynamicBatchServer(batch_size=4, timeout_ms=300)

# 서버를 별도의 스레드에서 실행
server_thread = threading.Thread(target=server.run)
server_thread.start()

# 실제 서비스처럼 비동기적으로 요청 추가
server.add_request({'id': 'A', 'target_len': 5, 'current_len': 0})
time.sleep(0.1) # A 도착 후 100ms 뒤에 B, C 도착
server.add_request({'id': 'B', 'target_len': 12, 'current_len': 0})
server.add_request({'id': 'C', 'target_len': 3, 'current_len': 0})

time.sleep(0.8) # 서버가 몇 스텝 진행하는 동안 대기
server.add_request({'id': 'D', 'target_len': 8, 'current_len': 0})
time.sleep(0.2)
server.add_request({'id': 'E', 'target_len': 6, 'current_len': 0})

# 모든 작업이 끝날 때까지 대기
server_thread.join()

[Client] >> 요청 A(5 토큰) 서버에 도착.

[Scheduler] >> 대기 큐에서 ['A']을(를) 배치에 추가!
--- 스텝 1 (현재 배치: ['A'], 최대 길이: 5) ---
[Client] >> 요청 B(12 토큰) 서버에 도착.
[Client] >> 요청 C(3 토큰) 서버에 도착.

[Scheduler] >> 대기 큐에서 ['B', 'C']을(를) 배치에 추가!
--- 스텝 2 (현재 배치: ['A', 'B', 'C'], 최대 길이: 12) ---
--- 스텝 3 (현재 배치: ['A', 'B', 'C'], 최대 길이: 12) ---
--- 스텝 4 (현재 배치: ['A', 'B', 'C'], 최대 길이: 12) ---

[Scheduler] << 요청 ['C'] 완료! 결과 반환.
--- 스텝 5 (현재 배치: ['A', 'B'], 최대 길이: 12) ---
[Client] >> 요청 D(8 토큰) 서버에 도착.

[Scheduler] << 요청 ['A'] 완료! 결과 반환.

[Scheduler] >> 대기 큐에서 ['D']을(를) 배치에 추가!
--- 스텝 6 (현재 배치: ['B', 'D'], 최대 길이: 12) ---
[Client] >> 요청 E(6 토큰) 서버에 도착.

[Scheduler] >> 대기 큐에서 ['E']을(를) 배치에 추가!
--- 스텝 7 (현재 배치: ['B', 'D', 'E'], 최대 길이: 12) ---
--- 스텝 8 (현재 배치: ['B', 'D', 'E'], 최대 길이: 12) ---
--- 스텝 9 (현재 배치: ['B', 'D', 'E'], 최대 길이: 12) ---
--- 스텝 10 (현재 배치: ['B', 'D', 'E'], 최대 길이: 12) ---
--- 스텝 11 (현재 배치: ['B', 'D', 'E'], 최대 길이: 12) ---
--- 스텝 12 (현재 배치: ['B', 'D', 'E'], 최대 길이: 12) ---

[Scheduler] << 요청 ['E'] 완료! 결과 반